In [4]:
import os, re, json, tqdm, csv, copy, math, gensim
import xml.etree.ElementTree as ET
import numpy as np
from nltk.probability import FreqDist
from scipy.special import kl_div, rel_entr
from gensim.models import Word2Vec, KeyedVectors

# Feature extraction

### Frequencies of compound, constituents and paraphrase

In [29]:
path_to_corpus = "/mount/resources/corpora/Royal-Society-Corpus/Royal_Society_Corpus_open_v6.0_texts_vrt/"
#badfiles = ["Royal_Society_Corpus_open_v6.0_text_rsta_1905_0011.vrt"]
def year_to_bin(year: int):
    if year < 1750:
        return 1
    elif year < 1800:
        return 2
    elif year < 1850:
        return 3
    elif year < 1900:
        return 4
    elif year < 1950:
        return 5
    else: return 6

In [30]:
timeslices = range(1, 7)
#my_compounds = ["glass tube", "fore part", "platinum wire", "copper wire", "iron wire", "salt solution", "blood pressure", "platinum black", "mineral manure", "induction coil", "mineral matter", "glass plate", "unit volume", "shoulder girdle", "absorption band", "cross section", "surface tension", "radius vector"]
my_compounds = [line.strip() for line in open("/home/users1/zabereus/zabereus/master_env/thesis/targets/compound_list_final.txt", "r")]
#topics = ['Agriculture', 'Anatomy 1', 'Anatomy 2', 'Astronomy', 'Atomic Physics', 'Biochemistry', 'Biography', 'Biology 1', 'Biology 2', 'Biology 3', 'Botany 1', 'Botany 2', 'Chemistry 1', 'Chemistry 2', 'Electricity', 'Fluid Dynamics', 'Formulae', 'Geography', 'Headmatter', 'Immunology', 'Measurement', 'Meteorology', 'Nervous System', 'Neurology', 'Optics', 'Paleontology', 'Physiology', 'Reporting', 'Tables', 'Thermodynamics']
topics = range(30)
lda_model = gensim.models.ldamodel.LdaModel.load("lda/lda.model")
id2word =  gensim.corpora.dictionary.Dictionary.load("lda/lda.model.id2word")

In [31]:
with open("ngram_dists_rsc.json", "r") as f:
    json_trigram_dist = json.load(f)
trigram_dist = {tuple(key.split()): value for key, value in json_trigram_dist.items()}
del json_trigram_dist

In [32]:
def get_surprisal(tokens: list):
    """
    Inverse log probability of last token given 3 tokens context
    """
    all_tetragram_probs = trigram_dist
    #print("surprisal of tokens: ", tokens)
    tokens = 3 * ["<null>"] + tokens
    prob = all_tetragram_probs[(tokens[-4], tokens[-3], tokens[-2])][tokens[-1]]
    return - math.log(prob)

use_annotated_surprisal = False

## Load existing features

In [41]:
with open("all_features.json", "r") as f:
    old_features = json.load(f)

In [34]:
feature_scheme = {"abs_freq_bin" + str(i): 0 for i in timeslices}
feature_scheme.update({"abs_para_freq_bin" + str(i): 0 for i in timeslices})
feature_scheme.update({"mod_surprisals": {slice: [] for slice in timeslices}, "head_surprisals": {slice: [] for slice in timeslices}, "years": [], "topics": {topic:0 for topic in topics}})
#TODO maybe make years a dict too

all_features = {c: copy.deepcopy(feature_scheme) for c in my_compounds}
#example_sentences = {comp:[] for comp in my_compounds}
all_constituents = set([const for c in my_compounds for const in c.split()])

const_feature_scheme = {"freq_bin" + str(i): 0 for i in timeslices}
#const_feature_scheme.update({"prodset_as_head": set(), "prodset_as_mod": set()})
all_constituent_features = {const: copy.deepcopy(const_feature_scheme) for const in all_constituents}
tokens_per_year = {y:0 for y in range(1665, 1997)}
tokens_per_topic = {topic:0 for topic in topics}

In [35]:
def get_sentences_from_file(fname, usePath = True):
    """
    Takes in a (badly) xml-formatted file and returns:
        - the year
        - the sentences as a list of lists of (token, POS, surprisal) str tuples
        using get_sentences_from_text
    """ 

    if usePath:
        fname = os.path.join(path_to_corpus, fname)
    with open(fname) as f:
        text = f.read()
    return get_sentences_from_text(text)

def get_sentences_from_text(text:str):    
    """
    Takes in a (ggf. badly formed) xml-string and returns:
        - the year
        - the sentences as a list of lists of (token, POS, surprisal) str tuples
    """
    text = re.sub(r'</?(page|inferred).*>\n', '', text)
    try:
        root = ET.fromstring(text)
    except ET.ParseError:
        with open("parse_errors.txt", "a") as f:
            f.write(text)
        raise ET.ParseError
    #if root.attrib['isAbstractOf'] != "" or root.attrib['language'] not in ['en', '']:
    if root.attrib['language'] not in ['en', '']: # filter out non-English texts, but include abstracts
        return None, None, [] #TODO hope I didn't break anything
    
    year = int(root.attrib['year'])
    
    # if 'primaryTopic' in root.attrib:
    #     primary_topic = root.attrib['primaryTopic']
    #     tokens_per_topic[primary_topic] += int(root.attrib["tokens"])
    # else: primary_topic = None
    
    just_tokens = []
    sentences = []
    for sentence in root:
        #assert sentence.tag == 's'
        lines = ''.join(sentence.itertext()).split('\n')
        tokens = []
        for line in lines:
            if len(line) > 0:
                token, pos, surp = line.split()[0], line.split()[1], line.split()[4]
                # just_tokens.append(token)
                # if not use_annotated_surprisal and token in all_constituent_features:
                #     surp = get_surprisal(just_tokens[-4:])
                tokens.append((token, pos, surp))
        sentences.append(tokens)
    
    #just_text = ' '.join(just_tokens)
    #bow = id2word.doc2bow(gensim.utils.simple_preprocess(just_text, deacc=True))
    #primary_topic = np.argmax([x[1] for x in lda_model[bow]])
    primary_topic = 0
    tokens_per_topic[primary_topic] += int(root.attrib["tokens"])
    tokens_per_year[year] += int(root.attrib["tokens"])

    return year, primary_topic, sentences

In [ ]:
def get_features_from_sents(sentences, year, topic):
    """
    Takes in sentences (of a document) as list of lists of (token, pos, surprisal) tuples
    and extracts features that get added to all_features and all_constituent_features
    """
    if not year:
        return
    bin = year_to_bin(year)
    global all_features, all_constituent_features
    for tokens in sentences:
        for i in range(len(tokens)):
            # count constituent occurrences (independent of whether they are part of a compound)
            if tokens[i][0] in all_constituent_features:
                all_constituent_features[tokens[i][0]]["freq_bin" + str(bin)] += 1
            if i == len(tokens)-1:
                break               # so we don't oob when checking the next token

            if tokens[i][1] == 'NN' and tokens[i+1][1] == 'NN' and (i==0 or tokens[i-1][1] != 'NN') and (i==len(tokens)-2 or tokens[i+2][1] != 'NN'):
                head = tokens[i+1][0].lower()
                modifier = tokens[i][0].lower()
                
                # For productivity
                # if head in all_constituent_features:
                #     all_constituent_features[head]["prodset_as_head"].add(modifier)
                # if modifier in all_constituent_features:
                #     all_constituent_features[modifier]["prodset_as_mod"].add(head)
                
                compound = modifier + " " + head
                if compound in my_compounds:
                    all_features[compound]["abs_freq_bin" + str(bin)] += 1
                    all_features[compound]["years"].append(year)
                    if use_annotated_surprisal:
                        mod_surprisal = float(tokens[i][2])
                        head_surprisal = float(tokens[i+1][2])
                    else:
                        try:
                            mod_surprisal = get_surprisal([token[0].lower().strip() for token in tokens[:i+1][-4:]])
                            head_surprisal = get_surprisal([token[0].lower().strip() for token in tokens[:i+2][-4:]])
                        except KeyError:
                            print(compound, [t[0] for t in tokens[:i+3]], i)
                            raise KeyError
                    all_features[compound]["mod_surprisals"][bin].append(mod_surprisal)
                    all_features[compound]["head_surprisals"][bin].append(head_surprisal)
                    # if compound == "salt water":
                    #     print(tokens[:i+1][-4:], float(tokens[i][2]))
                    #     print(tokens[:i+2][-4:], float(tokens[i+1][2]))
                    if topic:
                        all_features[compound]["topics"][topic] += 1
                    # if len(tokens) > 8 and len(tokens) < 30:
                    #     example_sentences[compound].append(" ".join([token[0] for token in tokens]))
                    #print(f"compound {compound} found!")

            # paraphrases
            #check: line start/not a noun, then NN, then of/in, then NN, then line ends/not a noun
            if i < len(tokens)-2 and (i==0 or tokens[i-1][1] != 'NN') and tokens[i][1] == 'NN' \
                and (tokens[i+1][0] == 'of' or tokens[i+1][0] == 'in') and tokens[i+2][1] == 'NN' \
                and (i == len(tokens)-3 or tokens[i+3][1] != 'NN'): 
                compound = tokens[i+2][0].lower() + " " + tokens[i][0].lower()
                bin = year_to_bin(year)
                if compound in all_features:
                    all_features[compound]["abs_para_freq_bin" + str(bin)] += 1
                
                    #print(f"para {compound} found!")

            # paraphrases with determiners
            #check: line start/not a noun, then NN, then of/in, then DET, then NN, then line ends/not a noun
            if i < len(tokens)-3 and (i==0 or tokens[i-1][1] != 'NN') and tokens[i][1] == 'NN' \
                and (tokens[i+1][0] == 'of' or tokens[i+1][0] == 'in') and tokens[i+2][1] == 'DT' and tokens[i+3][1] == 'NN' \
                and (i == len(tokens)-4 or tokens[i+4][1] != 'NN'): 
                compound = tokens[i+3][0].lower() + " " + tokens[i][0].lower()
                bin = year_to_bin(year)
                if compound in all_features:
                    all_features[compound]["abs_para_freq_bin" + str(bin)] += 1
                    #print(f"paradet {compound} found!")


In [38]:
def get_features_from_bigfile(bigfilename="/home/users1/zabereus/zabereus/master_env/RSC603Full.vrt", start= 0, end=50000):
    with open(bigfilename, 'r') as f:
        buffer = ""
        docs = 0
        for line in tqdm.tqdm(f):
            buffer += line + '\n'
            if "</text>" in line:
                if docs > start:
                    year, topic, sents = get_sentences_from_text(buffer)
                    get_features_from_sents(sents, year, topic)
                buffer = ""
                docs += 1
                if docs % 5000 == 0:
                    print(f"processed {docs} documents")
            if docs > end:
                break
            
                
get_features_from_bigfile()

0it [00:00, ?it/s]

13673194it [00:52, 271010.57it/s]

processed 5000 documents


35460260it [02:21, 259824.36it/s]

processed 10000 documents


...

258870781it [17:34, 254547.71it/s]

processed 40000 documents


299598349it [20:22, 246034.25it/s]

processed 45000 documents


323028123it [21:57, 245099.54it/s]


In [ ]:
# For self-made surprisal model
for compound, features in old_features.items():
    # average surprisal values
    for bin, surps in features["mod_surprisals"].items():
        features["avg_mod_surprisal_bin" + str(bin)] = 0 if len(surps) == 0 else np.round(sum(surps)/len(surps), 2)
    for bin, surps in features["head_surprisals"].items():
        features["avg_head_surprisal_bin" + str(bin)] = 0 if len(surps) == 0 else np.round(sum(surps)/len(surps), 2)
print(old_features["salt water"]["avg_head_surprisal_bin3"])

1.31


In [ ]:
no_occs = 0
for compound, features in all_features.items():
    occs = sum([occ for feat, occ in features.items() if feat.startswith('abs_freq')])
    para_occs = sum([occ for feat, occ in features.items() if feat.startswith('abs_para')])
    if occs + para_occs > 0:
        print(compound, f": Occurrs {occs} times, paraphrase occurs {para_occs} times")
    else: no_occs += 1
print(no_occs, " compounds do not appear in this sample")

adult form : Occurrs 172 times, paraphrase occurs 16 times
spiral form : Occurrs 131 times, paraphrase occurs 49 times
temperature curve : Occurrs 270 times, paraphrase occurs 41 times
frequency curve : Occurrs 218 times, paraphrase occurs 19 times
luminosity curve : Occurrs 185 times, paraphrase occurs 20 times
absorption curve : Occurrs 628 times, paraphrase occurs 12 times
spinode curve : Occurrs 79 times, paraphrase occurs 0 times
pressure curve : Occurrs 164 times, paraphrase occurs 18 times
glass tube : Occurrs 3333 times, paraphrase occurs 48 times
vacuum tube : Occurrs 677 times, paraphrase occurs 0 times
brass tube : Occurrs 676 times, paraphrase occurs 16 times
discharge tube : Occurrs 1422 times, paraphrase occurs 0 times
platinum tube : Occurrs 304 times, paraphrase occurs 17 times
side tube : Occurrs 586 times, paraphrase occurs 1 times
iron tube : Occurrs 235 times, paraphrase occurs 8 times
porcelain tube : Occurrs 194 times, paraphrase occurs 4 times
sea water : Occurrs

In [15]:
print(all_features["cut surface"])

{'abs_freq_bin1': 0, 'abs_freq_bin2': 7, 'abs_freq_bin3': 8, 'abs_freq_bin4': 81, 'abs_freq_bin5': 122, 'abs_freq_bin6': 123, 'abs_para_freq_bin1': 0, 'abs_para_freq_bin2': 0, 'abs_para_freq_bin3': 0, 'abs_para_freq_bin4': 0, 'abs_para_freq_bin5': 5, 'abs_para_freq_bin6': 9, 'surprisals': {1: [], 2: [0.41, 4.32, 3.12, 2.97, 3.47, 2.69, 2.97], 3: [3.13, 3.46, 3.46, 6.43, 3.07, 3.46, 3.46, 3.2], 4: [3.46, 4.12, 2.25, 7.55, 3.14, 2.97, 2.25, 2.44, 3.46, 6.2, 3.81, 6.2, 3.22, 2.96, 3.47, 3.47, 5.27, 3.22, 4.12, 2.94, 4.12, 2.94, 3.13, 6.47, 6.47, 2.22, 2.22, 6.47, 6.47, 3.81, 2.22, 0.44, 0.72, 0.44, 0.39, 0.63, 0.39, 6.43, 2.7, 3.13, 3.13, 3.13, 3.13, 2.94, 4.12, 1.7, 2.97, 4.65, 2.19, 1.7, 4.23, 2.97, 2.19, 2.69, 2.97, 2.19, 3.46, 3.14, 4.16, 4.16, 4.32, 2.19, 3.46, 2.97, 6.43, 2.19, 2.75, 2.75, 4.12, 3.46, 2.72, 3.46, 2.94, 3.2, 3.2, 2.44, 6.44, 4.12, 3.13, 2.19, 3.46], 5: [4.12, 4.12, 3.65, 3.46, 2.44, 6.43, 4.19, 4.19, 4.19, 3.81, 3.46, 2.96, 2.31, 3.23, 3.23, 2.31, 2.31, 2.42, 3.13, 3

### Secondary Processing

In [11]:
def kld_norm(occurrences, target_dist, topics = False):
    if topics:
        dist = [occs / sum(occurrences) for occs in occurrences] # normalize distribution
    else: # years
        dist = [0 for _ in range(1665, 1997)]
        for year in occurrences:
            dist[year-1665] += 1/len(occurrences) # normalize while adding

    #TODO we could also replace this implementation with a base 2 log one
    kld = sum(rel_entr(dist, target_dist))
    return 1 - math.exp(-kld)

In [ ]:
# Global distributions
tokens_per_year_dist = [occs / sum(tokens_per_year.values()) for occs in tokens_per_year.values()] # normalize distribution
tokens_per_topic_dist = [occs / sum(tokens_per_topic.values()) for occs in tokens_per_topic.values()] # normalize distribution
# tokens_per_topic_dist = json.load(open("distributions.json", "r"))[1]

tokens_per_bin = [0, 2.582856 + 3.414795, 6.342489, 9.112274, 36.993412, 65.431384, 172.018539] # all divided by 1e6 to give prettier feature numbers

# year and topic distribution per compound
for compound, features in all_features.items():
    # constituent frequencies
    modifier, head = compound.split()
    for bin in timeslices:
        # add constituent freqs, convert all freqs to relative frequencies
        features["head_freq_bin" + str(bin)] = all_constituent_features[head]["freq_bin" + str(bin)] / tokens_per_bin[bin]
        features["mod_freq_bin" + str(bin)] = all_constituent_features[modifier]["freq_bin" + str(bin)] / tokens_per_bin[bin]
        features["freq_bin" + str(bin)] = features["abs_freq_bin" + str(bin)] / tokens_per_bin[bin] 
        features["para_freq_bin" + str(bin)] = features["abs_para_freq_bin" + str(bin)] / tokens_per_bin[bin]

    # calculate divergences
    features["temp_divergence"] = kld_norm(features["years"], tokens_per_year_dist)
    features["topic_divergence"] = kld_norm(features["topics"].values(), tokens_per_topic_dist, topics=True)
    
    # average surprisal values
    for bin, surps in features["mod_surprisals"].items():
        features["avg_mod_surprisal_bin" + str(bin)] = 0 if len(surps) == 0 else sum(surps)/len(surps)
    for bin, surps in features["head_surprisals"].items():
        features["avg_head_surprisal_bin" + str(bin)] = 0 if len(surps) == 0 else sum(surps)/len(surps)
    #del features["years"], features["topics"], features["surprisals"]
    #print(features["temp_divergence"], features["topic_divergence"], features["avg_surprisal"])

In [18]:
print(all_features['cut surface'])
print(all_constituent_features['glass'])

{'abs_freq_bin1': 0, 'abs_freq_bin2': 7, 'abs_freq_bin3': 8, 'abs_freq_bin4': 81, 'abs_freq_bin5': 122, 'abs_freq_bin6': 123, 'abs_para_freq_bin1': 0, 'abs_para_freq_bin2': 0, 'abs_para_freq_bin3': 0, 'abs_para_freq_bin4': 0, 'abs_para_freq_bin5': 5, 'abs_para_freq_bin6': 9, 'surprisals': {1: [], 2: [0.41, 4.32, 3.12, 2.97, 3.47, 2.69, 2.97], 3: [3.13, 3.46, 3.46, 6.43, 3.07, 3.46, 3.46, 3.2], 4: [3.46, 4.12, 2.25, 7.55, 3.14, 2.97, 2.25, 2.44, 3.46, 6.2, 3.81, 6.2, 3.22, 2.96, 3.47, 3.47, 5.27, 3.22, 4.12, 2.94, 4.12, 2.94, 3.13, 6.47, 6.47, 2.22, 2.22, 6.47, 6.47, 3.81, 2.22, 0.44, 0.72, 0.44, 0.39, 0.63, 0.39, 6.43, 2.7, 3.13, 3.13, 3.13, 3.13, 2.94, 4.12, 1.7, 2.97, 4.65, 2.19, 1.7, 4.23, 2.97, 2.19, 2.69, 2.97, 2.19, 3.46, 3.14, 4.16, 4.16, 4.32, 2.19, 3.46, 2.97, 6.43, 2.19, 2.75, 2.75, 4.12, 3.46, 2.72, 3.46, 2.94, 3.2, 3.2, 2.44, 6.44, 4.12, 3.13, 2.19, 3.46], 5: [4.12, 4.12, 3.65, 3.46, 2.44, 6.43, 4.19, 4.19, 4.19, 3.81, 3.46, 2.96, 2.31, 3.23, 3.23, 2.31, 2.31, 2.42, 3.13, 3

### Context vectors

In [15]:
def sentence_reader_files(bin, file_start_index=0, file_end_index=100, compounds_only=False):
    """ Generator for RSC sentences from selected file range and time bin.
    if compounds_only = True, yields only sentences containing compounds from my_compounds
    """
    for file in os.listdir(path_to_corpus)[file_start_index:file_end_index]:
        year, _, sentences = get_sentences_from_file(file)
        if not year:
            continue
        document_bin = year_to_bin(year)
        if document_bin == bin:
            for tokens in sentences:
                compound_found = False
                for i in range(len(tokens)-1):
                    #replace compounds by underscore notation
                    if tokens[i][1] == 'NN' and tokens[i+1][1] == 'NN' and (i==0 or tokens[i-1][1] != 'NN') and (i==len(tokens)-2 or tokens[i+2][1] != 'NN'):
                        head = tokens[i+1][0].lower()
                        modifier = tokens[i][0].lower()
                        compound = modifier + " " + head
                        if compound in my_compounds:
                            tokens = tokens[:i] + [(f"{modifier}_{head}","","")] + tokens[i+2:] # replace constituents with compound
                            compound_found = True
                if compound_found or not compounds_only:
                    sent, _, _ = zip(*tokens)
                    yield sent

In [5]:
def sentence_reader_bigfile(bin, bigfilename="/home/users1/zabereus/zabereus/master_env/RSC603Full.vrt"):
    """ Generator for RSC sentences from selected time bin.
    """
    bin_ixs = [0, 3011, 4830, 7604, 14357, 24368, 47836]
    if bin == "all":
        start, end = bin_ixs[0], bin_ixs[-1] #entire dataset
    else:
        start, end = bin_ixs[bin-1], bin_ixs[bin]
    with open(bigfilename, 'r') as f:
        buffer = ""
        docs = 0
        for line in tqdm.tqdm(f):
            buffer += line # + '\n'
            if "</text>" in line:
                if docs > start:
                    year, _, sentences = get_sentences_from_text(buffer)
                    if not year:
                        buffer = ""
                        docs += 1
                        continue
                    # TODO not ideal implementation: if serveral compounds in 1 sent, only second gets underscored
                    for tokens in sentences:
                        mytokens = tokens
                        for i in range(len(tokens)-1):
                            #replace compounds by underscore notation
                            if tokens[i][1] == 'NN' and tokens[i+1][1] == 'NN' and (i==0 or tokens[i-1][1] != 'NN') and (i==len(tokens)-2 or tokens[i+2][1] != 'NN'):
                                head = tokens[i+1][0].lower()
                                modifier = tokens[i][0].lower()
                                compound = modifier + " " + head
                                if compound in my_compounds:
                                    mytokens = tokens[:i] + [(f"{modifier}_{head}","","")] + tokens[i+2:] # replace constituents with compound
                        sent, _, _ = zip(*mytokens)
                        yield sent
                buffer = ""
                docs += 1
            if docs >= end:
                break

In [10]:
vectors = Word2Vec(list(sentence_reader_bigfile("all")), min_count=1, workers=10).wv
vectors.save(f"w2v_total/word2vec.wordvectors")

0it [00:00, ?it/s]

323001660it [36:47, 146329.18it/s]


In [ ]:
bin_models = []
for bin in tqdm.tqdm(timeslices):
    bin_models.append(Word2Vec(list(sentence_reader_bigfile(bin)), min_count=1))

In [17]:
mod_cos_avgs = [0.44416894231523785, 0.4160444166649271, 0.36184621912084125, 0.36857576150643195, 0.41417080549738156, 0.35760397040630104]
head_cos_avgs = [0.306751651290272, 0.3844731435857036, 0.36531950668676905, 0.3818672882180895, 0.45482586114070356, 0.38456483871956904]
mod_cos_avgs2 = [0.4220802081482751, 0.3996399021535008, 0.350640249938557, 0.36421989694629847, 0.41394795769204695, 0.366035366908859] 
head_cos_avgs2 = [0.30476378715996233, 0.35924903117120266, 0.34632398040082896, 0.3743804204619017, 0.4565632594635901, 0.392133777168556]
mod_cos_avgs3 = [0.42606148097131935, 0.39534496602222874, 0.35452830227287974, 0.36812203932480003, 0.40691615990721264, 0.35605572208037806] 
head_cos_avgs_3 = [0.3219953633711806, 0.3664323656735095, 0.3494691948547322, 0.38460240639863214, 0.45352761117884743, 0.3903344878757486]
for vec in ["", "2", "3"]:
    mod_avg, head_avg = {"": (mod_cos_avgs, head_cos_avgs), "2": (mod_cos_avgs2, head_cos_avgs2), "3": (mod_cos_avgs3, head_cos_avgs_3)}[vec]
    for bin in timeslices:
        if vec == "":
            vectors = Word2Vec.load(f"w2v/word2vec_bin{bin}.model").wv
        else:
            vectors = KeyedVectors.load(f"w2v{vec}/word2vec_bin{bin}.wordvectors")
        for compound in all_features.keys():
            mod, head = compound.split()
            if mod + "_" + head in vectors:
                all_features[compound][f"cosine{vec}_mod_bin{bin}"] = float(vectors.similarity(mod + "_" + head, mod)) if mod in vectors else mod_avg[bin-1]
                all_features[compound][f"cosine{vec}_head_bin{bin}"] = float(vectors.similarity(mod + "_" + head, head)) if head in vectors else head_avg[bin-1]
            else:
                all_features[compound][f"cosine{vec}_mod_bin{bin}"] = mod_avg[bin-1]
                all_features[compound][f"cosine{vec}_head_bin{bin}"] = head_avg[bin-1]

In [ ]:
# Get average cosine similarities for use as default values
all_mod_cosines = {bin: [] for bin in timeslices}
all_head_cosines = {bin: [] for bin in timeslices}

for bin in timeslices:
    vectors = KeyedVectors.load(f"w2v2/word2vec_bin{bin}.wordvectors")
    for compound in all_features.keys():
        mod, head = compound.split()
        if mod + "_" + head in vectors and mod in vectors:
            all_mod_cosines[bin].append(float(vectors.similarity(mod + "_" + head, mod)))
        if mod + "_" + head in vectors and head in vectors:
            all_head_cosines[bin].append(float(vectors.similarity(mod + "_" + head, head)))

mod_cos_avgs = [sum(sims)/len(sims) for sims in all_mod_cosines.values()]
head_cos_avgs = [sum(sims)/len(sims) for sims in all_head_cosines.values()]
print(mod_cos_avgs, head_cos_avgs)

[0.4220802081482751, 0.3996399021535008, 0.350640249938557, 0.36421989694629847, 0.41394795769204695, 0.366035366908859] [0.30476378715996233, 0.35924903117120266, 0.34632398040082896, 0.3743804204619017, 0.4565632594635901, 0.392133777168556]


## N-gram surprisal model

In [26]:
def get_sentences_from_text_fast(text:str):    
    """
    Takes in a (ggf. badly formed) xml-string and returns:
        - the sentences as a list of lists of (token, POS, surprisal) str tuples
    """
    text = re.sub(r'</?(page|inferred).*>\n', '', text)
    try:
        root = ET.fromstring(text)
    except ET.ParseError:
        with open("parse_errors.txt", "a") as f:
            f.write(text)
        raise ET.ParseError
    if root.attrib['language'] not in ['en', '']: # filter out non-English texts
        return []

    sentences = []
    for sentence in root:
        #assert sentence.tag == 's'
        lines = ''.join(sentence.itertext()).split('\n')
        tokens = []
        for line in lines:
            if len(line) > 0:
                token = line.split()[0]
                tokens.append(token)
        sentences.append(tokens)
    return sentences

def get_all_contexts(bigfilename="/home/users1/zabereus/zabereus/master_env/RSC603Full.vrt", start=0, end=50000):
    """
    idea: Iterate through corpus once, grabbing all contexts that occur before our compounds
    Then iterate a second time, grabbing all instances of those contexts
    """
    occuring_trigram_contexts = []
    with open(bigfilename, 'r') as f:
        buffer = ""
        docs = 0
        for line in tqdm.tqdm(f):
            buffer += line.lower() + '\n'
            if "</text>" in line:
                if docs > start:
                    sents = get_sentences_from_text_fast(buffer)
                    for sent in sents:
                        trigram_contexts = get_compound_contexts_from_sent(sent)
                        occuring_trigram_contexts.extend(trigram_contexts)
                buffer = ""
                docs += 1
                if docs % 5000 == 0:
                    print(f"processed {docs} documents")
            if docs > end:
                break
    occuring_trigram_contexts = set(occuring_trigram_contexts)
    print(f"{len(occuring_trigram_contexts)} trigram contexts occur.")
    return occuring_trigram_contexts


def get_compound_contexts_from_sent(tokens):
    """
    Takes in a sentence as list of tokens and retuns all contexts that occur before a target compound
    """
    trigram_contexts = []
    tokens = 3 * ["<null>"] + tokens + ["<null>"]

    sentence_5grams = [(tokens[i], tokens[i+1], tokens[i+2], tokens[i+3], tokens[i+4]) for i in range(len(tokens) - 4)]

    for fivegram in sentence_5grams:
        if fivegram[3] in all_constituent_features: #for speed
            if fivegram[2] + " " + fivegram[3] in my_compounds or fivegram[3] + " " + fivegram[4] in my_compounds:
                trigram_contexts.append((fivegram[0], fivegram[1], fivegram[2]))

    return trigram_contexts
                    
def get_ngram_probs(tris, bigfilename="/home/users1/zabereus/zabereus/master_env/RSC603Full.vrt", start=0, end=50000):
    trigram_probs ={trigram: {"other": 0} for trigram in tris}
    with open(bigfilename, 'r') as f:
        buffer = ""
        docs = 0
        for line in tqdm.tqdm(f):
            buffer += line.lower() + '\n'
            if "</text>" in line:
                if docs > start:
                    sents = get_sentences_from_text_fast(buffer)
                    for sent in sents:
                        get_contexts_from_sent(sent + ["<eos>"], trigram_probs)
                buffer = ""
                docs += 1
                if docs % 5000 == 0:
                    print(f"processed {docs} documents")
            if docs > end:
                break
    return normalize_dist(trigram_probs)

def get_contexts_from_sent(tokens, trigram_probs):
    """
    Takes in a sentence as list of tokens (incl <eos> at the end) and updates context probabilities
    """
    tokens = 3 * ["<null>"] + tokens
    sentence_tetragrams = [(tokens[i], tokens[i+1], tokens[i+2], tokens[i+3]) for i in range(len(tokens) - 3)]
    trigrams = trigram_probs.keys()

    for tetragram in sentence_tetragrams:
        if (tetragram[0], tetragram[1], tetragram[2]) in trigrams:
            if tetragram[3] in all_constituents:
                if tetragram[3] in trigram_probs[(tetragram[0], tetragram[1], tetragram[2])]:
                    trigram_probs[(tetragram[0], tetragram[1], tetragram[2])][tetragram[3]]
                else:
                    trigram_probs[(tetragram[0], tetragram[1], tetragram[2])][tetragram[3]] = 1
            else:
                trigram_probs[(tetragram[0], tetragram[1], tetragram[2])]["other"] += 1    

            
def normalize_dist(dist: dict):
    return {context: {token: prob/sum(probdist.values()) if sum(probdist.values()) else 0 for token, prob in probdist.items()} for context, probdist in dist.items()}


In [19]:
trigram_contexts = get_all_contexts()

13645591it [00:31, 443233.78it/s]

processed 5000 documents


35518608it [01:25, 427199.41it/s]

processed 10000 documents


67012967it [02:40, 433298.72it/s]

...

processed 40000 documents


299645268it [11:56, 439438.64it/s]

processed 45000 documents


323028123it [12:50, 419095.61it/s]


124333 trigram contexts occur.


In [28]:
trigram_dist = get_ngram_probs(trigram_contexts)
json_trigram_dist = {" ".join(key): value for key, value in trigram_dist.items()}
with open("ngram_dists_rsc.json", "w") as f:
    json.dump(json_trigram_dist, f)

0it [00:00, ?it/s]

13658947it [00:34, 388276.48it/s]

processed 5000 documents


35511138it [01:31, 388399.82it/s]

processed 10000 documents


...

258898758it [10:59, 414387.36it/s]

processed 40000 documents


299602291it [12:39, 418198.27it/s]

processed 45000 documents


323028123it [13:37, 395236.51it/s]


## Neighborhood density

In [7]:
averages = [0.6798875188188894, 0.7088567922332069, 0.6867188171963942, 0.6971930293668628, 0.6874099814891815, 0.6754843184399225]
for bin in timeslices:
    vectors = KeyedVectors.load(f"w2v2/word2vec_bin{bin}.wordvectors")
    for compound in all_features.keys():
        compound_ = "_".join(compound.split())
        if compound_ in vectors:
            most_similar = vectors.most_similar(positive=compound_, topn=10)
            avg = np.average([token[1] for token in most_similar])
        else:
            avg = averages[bin-1]
        all_features[compound][f"neighborhood_density_bin{bin}"] = avg  #high similarity means high density
print(all_features["salt water"]["neighborhood_density_bin3"])
print(all_features["salt water"]["neighborhood_density_bin6"])
print(all_features["luminosity curve"]["neighborhood_density_bin3"])
print(all_features["luminosity curve"]["neighborhood_density_bin6"])

0.7010184586048126
0.7350541472434997
0.6867188171963942
0.5645894825458526


In [6]:
# get averages
neighborhood_sims = []
for bin in timeslices:
    sims_slices = []
    vectors = KeyedVectors.load(f"w2v2/word2vec_bin{bin}.wordvectors")
    for compound in all_features.keys():
        compound_ = "_".join(compound.split())
        if compound_ in vectors:
            most_similar = vectors.most_similar(positive=compound_, topn=10)
            avg = np.average([token[1] for token in most_similar])
            sims_slices.append(avg)
    print(np.average(sims_slices))
    neighborhood_sims.append(np.average(sims_slices))


0.6798875188188894
0.7088567922332069
0.6867188171963942
0.6971930293668628
0.6874099814891815
0.6754843184399225


## Saving all features

In [20]:
with open("all_features.json", "w") as f:
    json.dump(all_features, f)

for feats in all_constituent_features.values():
    del feats["prodset_as_head"], feats["prodset_as_mod"]
print(all_constituent_features["glass"])

with open("all_constituent_features.json", "w") as f:
    json.dump(all_constituent_features, f)

with open("distributions.json", "w") as exs_f:
    json.dump([tokens_per_year_dist, tokens_per_topic_dist], exs_f)

{'freq_bin1': 402, 'freq_bin2': 3143, 'freq_bin3': 4830, 'freq_bin4': 11804, 'freq_bin5': 15513, 'freq_bin6': 13884}


In [48]:
with open("all_features_newsurps.json", "w") as f:
    json.dump(old_features, f)